## Installation
Install the decent-bench package and PyTorch if not already present.

In [ ]:
%pip install -e ../../.
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128

## Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch
from src.dataset import NIMDatasetHandler
from src.nim_helpers import NimModel, ThresholdSigmoid, create_heatmap_plots
from src.visualize_dataset import visualize_nim_dataset
from torch import nn

import decent_bench.algorithms.p2p as dec_algorithms
import decent_bench.utils.interoperability as iop
from decent_bench import benchmark
from decent_bench.agents import Agent
from decent_bench.algorithms.utils import pytorch_initialization
from decent_bench.costs import PyTorchCost
from decent_bench.metrics import metric_library as ml
from decent_bench.metrics.metric_utils import single
from decent_bench.networks import P2PNetwork
from decent_bench.schemes import GaussianNoise, TopK, UniformActivationRate, UniformDropRate
from decent_bench.utils.checkpoint_manager import CheckpointManager
from decent_bench.utils.types import SupportedDevices


## Experiment parameters

In [ ]:
checkpoint_path = Path("results/nim")
iterations = 20_000
n_trials = 3
state_snapshot_period = 500
n_agents = 5
n_neighbors = 4
samples_per_partition = 25_000  # Set to None to use all samples in the dataset partition
overlap = 0.1
label_balance = 2.0  # Ratio of label samples, set to 1.0 for truly balanced labels. See dataset.py for details.
image_file = "data/kth_floorplan_sample.png"
batch_size = 512
test_samples = 50_000
device = SupportedDevices.CPU  # Set to GPU (or MPS on mac) if avaiable as it is most likely faster for this benchmark

# Define network impairments, set to None to disable
activation = UniformActivationRate(0.8)
noise = GaussianNoise(0.0, 0.01)
compression = TopK(0.1)
drops = UniformDropRate(0.2)

# Set seed
iop.set_seed(47)

## Define which metrics to calculate

In [ ]:
table_metrics = [
    ml.ConsensusError([min, np.average, max]),
    ml.GradientCalls([np.average, sum]),
    ml.SentMessages([np.average, sum]),
    ml.ReceivedMessages([np.average, sum]),
    ml.SentMessagesDropped([np.average, sum]),
    ml.GradientNorm([single]),
    ml.MSE([min, np.average, max]),
    ml.Loss([min, np.average, max]),
]

plot_metrics = [
    [ml.ConsensusError([], x_log=False, y_log=True)],
    [ml.MSE([], x_log=False, y_log=True)],
    [ml.Loss([], x_log=False, y_log=False)],
]

## Define PyTorch model generator

In [ ]:
def model_generator() -> torch.nn.Module:
    """Generate a model for the NIM dataset."""
    return NimModel(
        input_size=2,  # Input is (x, y) coordinates
        hidden_sizes=[256, 64, 64],
        output_size=1,  # Output is a single value representing the probability of being in a wall
    )

## Create Neural Implicit Mapping datasets

In [ ]:
train_dataset = NIMDatasetHandler(
    image_file=image_file,
    n_partitions=n_agents,
    samples_per_partition=samples_per_partition,
    label_balance=label_balance,
    overlap=overlap,
)
test_data = train_dataset.get_test_set(label_balance=1.0, num_samples=test_samples)

## Visualize dataset
Shows 1'000 random samples from each partition

In [ ]:
visualize_nim_dataset(train_dataset, num_samples_per_partition=1000)
plt.show()

## Create checkpoint manager

In [ ]:
cm = CheckpointManager(
    checkpoint_dir=checkpoint_path,
    checkpoint_step=None,
    keep_n_checkpoints=1,
    benchmark_metadata={
        "dataset": "NIM",
        "n_agents": n_agents,
        "n_neighbors": n_neighbors,
        "drops": drops.__class__.__name__ if drops else None,
        "activity": activation.__class__.__name__ if activation else None,
        "compression": compression.__class__.__name__ if compression else None,
        "noise": noise.__class__.__name__ if noise else None,
    },
)

## Create costs, agents, network, and benchmark problem

In [ ]:
costs = [
    PyTorchCost(
        dataset=p,
        model=model_generator(),
        # BCEWithLogitsLoss combines a sigmoid layer and the BCELoss in one single class, which is more numerically
        # stable than using a plain Sigmoid followed by a BCELoss. If you want to use a different loss function,
        # double check if you need to add a Sigmoid layer to the model generator.
        loss_fn=nn.BCEWithLogitsLoss(),
        # If you want to use a threshold-based activation instead of sigmoid, uncomment the line below and comment
        # out the sigmoid line. Note that the threshold should be chosen based on the expected output range of
        # the model.
        # final_activation=ThresholdSigmoid(threshold=0.5),
        final_activation=nn.Sigmoid(),
        batch_size=batch_size,
        max_batch_size=batch_size,
        device=device,
    )
    for p in train_dataset.get_partitions()
]
agents = [
    Agent(
        i,
        cost,
        state_snapshot_period=state_snapshot_period,
        activation=activation,
    )
    for i, cost in enumerate(costs)
]
graph = nx.random_regular_graph(d=n_neighbors, n=n_agents, seed=iop.get_seed())
network = P2PNetwork(
    graph=graph,
    agents=agents,
    message_noise=noise,
    message_compression=compression,
    message_drop=drops,
)
problem = benchmark.BenchmarkProblem(
    network=network,
    test_data=test_data,
)

## Create algorithms

In [ ]:
x0 = pytorch_initialization(network, all_same=True)
algorithms = [
    dec_algorithms.DGD(
        step_size=0.1,
        iterations=iterations,
        x0=x0,
    ),
    dec_algorithms.KGT(
        iterations=iterations,
        num_local_steps=10,
        step_size=0.025,
        aux_step_size=0.5,
        x0=x0,
    ),
    dec_algorithms.ProxSkip(
        iterations=iterations,
        step_size=0.05,
        aux_step_size=0.05,
        comm_probability=0.2,
        chi=1.0,
        x0=x0,
    ),
    dec_algorithms.LED(
        iterations=iterations,
        num_local_steps=10,
        step_size=0.025,
        aux_step_size=0.025,
        x0=x0,
    ),
    dec_algorithms.LT_ADMM(
        iterations=iterations,
        num_local_steps=10,
        step_size=0.025,
        aux_step_size=0.025,
        penalty=1.0,
        x0=x0,
    ),
]

## Start benchmark execution
Note: If you have many samples per agent, it might take a while to start.

In [ ]:
result = benchmark.benchmark(
    algorithms=algorithms,
    benchmark_problem=problem,
    n_trials=n_trials,
    show_speed=True,
    show_trial=True,
    checkpoint_manager=cm,
)

metric_result = benchmark.compute_metrics(
    benchmark_result=result,
    checkpoint_manager=cm,
    table_metrics=table_metrics,
    plot_metrics=plot_metrics,
)

benchmark.display_metrics(
    metrics_result=metric_result,
    checkpoint_manager=cm,
)

figs = create_heatmap_plots(
    image_file=image_file,
    result=result,
    save_path=cm.get_results_path(),
)

## Optionally resume benchmark from checkpoint
Uncomment the code in the code block below to resume an interrupted benchmark execution

In [ ]:
# result = benchmark.resume_benchmark(
#     checkpoint_manager=cm,
#     create_backup=False,
#     show_speed=True,
#     show_trial=True,
# )

# metric_result = benchmark.compute_metrics(
#     benchmark_result=result,
#     checkpoint_manager=cm,
#     table_metrics=table_metrics,
#     plot_metrics=plot_metrics,
# )

# benchmark.display_metrics(
#     metrics_result=metric_result,
#     checkpoint_manager=cm,
# )

# figs = create_heatmap_plots(
#     image_file=image_file,
#     result=result,
#     save_path=cm.get_results_path(),
# )